# 02 — Train & compare flood models (ConvLSTM · CNN-LSTM · ResNet)

Trains three architectures on the same task, the same way, and compares them.

**Inputs** (per sample): the D−1 GOES sequence `(T=6, N_CH=7, 1500, 2500)` — 6 ABI bands
**+ a per-frame lead-time channel (the "time")** — plus a learned **per-cell location
embedding** seeded from climatology (the "where").
**Output:** per-cell flood logits `(59, 95)` → `sigmoid` = P(flood) on the 50 km grid.

| model | how it fuses the 6 frames |
|---|---|
| **ConvLSTM** | spatial recurrence — a 2-D hidden state walked over T |
| **CNN-LSTM** | per-frame CNN → pool to cells → a **per-cell LSTM** over T |
| **ResNet** | no recurrence — stack the T frames as channels → residual CNN |

All three share the **CellPool** (exact GOES-pixel→cell regrid), the **LocHead**
(per-cell embedding + climatology-seeded bias), and the **recall-favoring Tversky loss**.
Only the spatio-temporal backbone differs — that's the comparison.

**Compute.** Each model trains across **both GPUs with true DDP** (balanced, via
`train_compare.py` launched with `torchrun`), bf16 + TF32, batch `BATCH_PER_GPU`/card.
The model classes live in `floodnet.py` (so the DDP workers can import them); we show
their source below. After **each** model: training curve, test PR-AUC / ROC-AUC / F1, and
sample test-day prediction maps. A final section overlays all three.

## 1. Setup

In [ ]:
import sys, time, subprocess, inspect
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_recall_curve, roc_curve,
                             classification_report, f1_score)

ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
MODEL_DIR = ROOT / "notebooks" / "model"
sys.path.insert(0, str(MODEL_DIR))
from config import BATCH_PER_GPU, CKPT_DIR, N_CH
import floodnet
from floodnet import COMPARE_MODELS, build_pix2cell, pool_sub

EPOCHS = 5                                          # per model
COMPARE_DIR = CKPT_DIR / "compare"
LABELS = {"convlstm": "ConvLSTM", "cnnlstm": "CNN-LSTM", "resnet": "ResNet"}
COL = {"convlstm": "#6a4c93", "cnnlstm": "#1d6fb8", "resnet": "#e09f3e"}

# grid + land mask for plotting (regenerated from config; pix2cell is cached)
p2c, GRID_R, GRID_C, land = build_pix2cell()
print(f"grid {GRID_R}x{GRID_C} | land cells {int(land.sum())} | N_CH {N_CH} | "
      f"GPUs {torch.cuda.device_count()} | batch {BATCH_PER_GPU}/card")

## 2. The three models

Defined in `floodnet.py` (importable so the DDP workers can build them). Each maps
`x (B, T, 7, H, W)` → per-cell logits `(B, 59, 95)`; they differ only in the backbone,
then share `CellPool` + `LocHead`. Their source, and parameter counts:

In [ ]:
for cls in (floodnet.ConvLSTMNet, floodnet.CNNLSTMNet, floodnet.ResNetNet):
    print(inspect.getsource(cls))

# parameter counts (build on CPU with a dummy climatology)
sub = pool_sub(p2c); dummy = np.zeros((GRID_R, GRID_C), np.float32)
for key, cls in COMPARE_MODELS.items():
    m = cls(sub, GRID_R, GRID_C, dummy)
    print(f"{LABELS[key]:<9} params {sum(p.numel() for p in m.parameters()):,}")
    del m

## 3. Train all three — DDP across both GPUs

Each model is trained by `train_compare.py` under `torchrun --nproc_per_node=2` (true
DDP, balanced over both cards). It uses a **temporal** split (earliest 70% train / next
10% val / last 20% test), bf16 + TF32, and saves test predictions + training history to
`compare/{model}.npz`. Output streams below; re-running skips models whose `.npz` exists.

In [ ]:
def train(model, epochs=EPOCHS, force=False):
    out = COMPARE_DIR / f"{model}.npz"
    if out.exists() and not force:
        print(f"[{model}] cached -> {out.name} (force=True to retrain)"); return
    cmd = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node=2",
           "train_compare.py", "--model", model, "--epochs", str(epochs)]
    print(f"$ {' '.join(cmd[-5:])}", flush=True)
    subprocess.run(cmd, cwd=str(MODEL_DIR), check=True)

t0 = time.perf_counter()
for m in ["convlstm", "cnnlstm", "resnet"]:
    train(m)
print(f"\nall models ready in {(time.perf_counter()-t0)/60:.1f} min")

## 4. Comparison helpers

Load each model's saved test predictions and define one `show(model)` that draws its
training curve, PR/ROC curves, classification report, and sample prediction maps.

In [ ]:
R = {m: dict(np.load(COMPARE_DIR / f"{m}.npz", allow_pickle=False))
     for m in ["convlstm", "cnnlstm", "resnet"]}
trues = R["convlstm"]["trues"]
y = trues[:, land].ravel().astype(int)
BASE = y.mean()
dates = R["convlstm"]["dates"]
print(f"test days {trues.shape[0]} | flood base rate {BASE:.3%}")


def best_f1(yt, p):
    return max((f1_score(yt, p > t, zero_division=0), t)
               for t in np.quantile(p, np.linspace(0.90, 0.999, 40)))


def metrics(m):
    p = R[m]["probs"][:, land].ravel()
    ap, roc = average_precision_score(y, p), roc_auc_score(y, p)
    f1, thr = best_f1(y, p)
    return dict(ap=ap, roc=roc, f1=f1, thr=thr)


def show(m):
    d = R[m]; p = d["probs"][:, land].ravel(); mt = metrics(m)
    print(f"=== {LABELS[m]} | params {int(d['params']):,} ===")
    print(f"PR-AUC {mt['ap']:.4f} ({mt['ap']/BASE:.1f}x base)  ROC {mt['roc']:.3f}  "
          f"bestF1 {mt['f1']:.3f}")
    print(classification_report(y, p > mt["thr"],
                                target_names=["no flood", "flood"], digits=3))
    # training curve + PR + ROC
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(d["hist_epoch"], d["hist_train"], "o-", label="train loss")
    a2 = ax[0].twinx(); a2.plot(d["hist_epoch"], d["hist_val"], "s--", color="green",
                                label="val PR-AUC")
    ax[0].set_xlabel("epoch"); ax[0].set_ylabel("Tversky loss"); a2.set_ylabel("val PR-AUC")
    ax[0].set_title(f"{LABELS[m]}: training"); ax[0].legend(loc="upper left")
    a2.legend(loc="upper right")
    pr, rc, _ = precision_recall_curve(y, p)
    ax[1].plot(rc, pr, color=COL[m]); ax[1].axhline(BASE, color="k", ls="--", lw=1)
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")
    ax[1].set_title(f"PR (AUC {mt['ap']:.3f}, {mt['ap']/BASE:.1f}x base)")
    fpr, tpr, _ = roc_curve(y, p)
    ax[2].plot(fpr, tpr, color=COL[m]); ax[2].plot([0, 1], [0, 1], "k--", lw=1)
    ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].set_title(f"ROC (AUC {mt['roc']:.3f})")
    plt.tight_layout(); plt.show()
    # sample prediction maps — 6 busiest test days
    nfl = trues[:, land].sum(1); pick = np.argsort(nfl)[::-1][:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), constrained_layout=True)
    for ax_, i in zip(axes.flat, pick):
        truth = trues[i].astype(bool)
        ax_.imshow(np.where(land, d["probs"][i], np.nan), cmap="viridis", vmin=0, vmax=1)
        yy, xx = np.where(truth & land)
        ax_.scatter(xx, yy, s=10, facecolors="none", edgecolors="red", linewidths=0.9)
        f1d = f1_score(truth[land].ravel(), (d["probs"][i] > mt["thr"])[land].ravel(),
                       zero_division=0)
        ax_.set_title(f"{dates[i]}  F1={f1d:.2f}", fontsize=9)
        ax_.set_xticks([]); ax_.set_yticks([])
    fig.suptitle(f"{LABELS[m]} — P(flood) heatmap, red = true flood cells", fontsize=12)
    plt.show()

## 5. Model A — ConvLSTM

In [ ]:
show("convlstm")

## 6. Model B — CNN-LSTM

In [ ]:
show("cnnlstm")

## 7. Model C — ResNet

In [ ]:
show("resnet")

## 8. Final comparison — all three models

Same temporal test split, same loss, same location head — only the backbone differs.

In [ ]:
models = ["convlstm", "cnnlstm", "resnet"]
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
for m in models:
    p = R[m]["probs"][:, land].ravel()
    pr, rc, _ = precision_recall_curve(y, p); ax[0].plot(rc, pr, label=LABELS[m], color=COL[m], lw=2)
    fpr, tpr, _ = roc_curve(y, p); ax[1].plot(fpr, tpr, label=LABELS[m], color=COL[m], lw=2)
ax[0].axhline(BASE, color="k", ls="--", lw=1, label=f"base {BASE:.2%}")
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("PR (test)"); ax[0].legend()
ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].set_title("ROC (test)"); ax[1].legend()
M = {m: metrics(m) for m in models}
x = np.arange(len(models)); w = 0.25
ax[2].bar(x - w, [M[m]["ap"] / BASE for m in models], w, label="PR-AUC / base", color="#6a4c93")
ax[2].bar(x, [M[m]["roc"] for m in models], w, label="ROC-AUC", color="#1d6fb8")
ax[2].bar(x + w, [M[m]["f1"] for m in models], w, label="best F1", color="#e09f3e")
ax[2].set_xticks(x); ax[2].set_xticklabels([LABELS[m] for m in models])
ax[2].set_title("test metrics"); ax[2].legend()
plt.tight_layout(); plt.show()

print(f"{'model':<10}{'params':>10}{'PR-AUC':>9}{'xbase':>7}{'ROC':>7}{'F1':>7}")
for m in models:
    print(f"{LABELS[m]:<10}{int(R[m]['params']):>10,}{M[m]['ap']:>9.4f}"
          f"{M[m]['ap']/BASE:>6.1f}x{M[m]['roc']:>7.3f}{M[m]['f1']:>7.3f}")